In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# Edit DATA_PATH to point to your CSV file.
# All other paths are derived automatically.
from pathlib import Path

DATA_PATH = Path.cwd() / "data" / "spotify-tracks-dataset.csv"
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd().parent / "data" / "spotify-tracks-dataset.csv"
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd() / "backend" / "data" / "spotify-tracks-dataset.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at '{DATA_PATH.resolve()}'.\n"
        "Update DATA_PATH above to point to the correct location."
    )

print("Dataset path:", DATA_PATH.resolve())

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# ── Column overview ────────────────────────────────────────────────────────
print("Columns:")
print(list(df.columns))

In [ ]:
# ── Drop artefact index columns ────────────────────────────────────────────
unnamed_cols = [c for c in df.columns if c.startswith("Unnamed")]
df.drop(columns=unnamed_cols, inplace=True)
print(f"Dropped {len(unnamed_cols)} unnamed column(s): {unnamed_cols}")
print(f"Shape after drop: {df.shape}")

In [ ]:
# ── Null value analysis ────────────────────────────────────────────────────
null_counts = df.isnull().sum()
print(f"Total nulls: {null_counts.sum()}")
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "No nulls found.")

null_pct = (null_counts / len(df) * 100).round(2)
print("\nNull % per column:")
print(null_pct[null_pct > 0] if null_pct.sum() > 0 else "None.")

In [ ]:
# ── Drop nulls and duplicates ──────────────────────────────────────────────
before = len(df)
df.dropna(inplace=True)
print(f"Dropped {before - len(df)} rows with nulls.")

before = len(df)
df.drop_duplicates(inplace=True)
print(f"Dropped {before - len(df)} duplicate rows.")
print(f"Shape after cleaning: {df.shape}")

In [ ]:
# ── Drop non-numeric identifier columns ────────────────────────────────────
text_cols = ["track_id", "artists", "album_name", "track_name"]
existing  = [c for c in text_cols if c in df.columns]
df.drop(columns=existing, inplace=True)
print(f"Dropped text columns: {existing}")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# ── Remove rows with physically impossible values ──────────────────────────
broken_mask = (
    (df["duration_ms"]    <= 0)
    | (df["tempo"]         <= 0)
    | (df["loudness"]      > 0)
    | (df["time_signature"] < 3)
)
before = len(df)
df = df.loc[~broken_mask].copy()
print(f"Removed {before - len(df)} rows with invalid values.")
print(f"Final shape: {df.shape}")

In [ ]:
# ── Final dataset info ─────────────────────────────────────────────────────
df.info()
print("\nBasic statistics:")
df.describe().round(3)

In [ ]:
# ── Genre distribution ─────────────────────────────────────────────────────
print(f"Unique genres: {df['track_genre'].nunique()}")
top_genres = df["track_genre"].value_counts().head(20)

fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(x=top_genres.index, y=top_genres.values, palette="viridis", ax=ax)
ax.set_title("Top 20 Genres by Track Count", fontsize=13, fontweight="bold")
ax.set_xlabel("Genre")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── Numeric feature distributions ─────────────────────────────────────────
numeric_cols = df.select_dtypes(include="number").columns.tolist()

n_cols = 4
n_rows = -(-len(numeric_cols) // n_cols)  # ceiling division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=50, color="#3498db",
                 edgecolor="white", linewidth=0.3)
    axes[i].set_title(col, fontsize=9, fontweight="bold")
    axes[i].set_xlabel("")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Numeric Feature Distributions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ────────────────────────────────────────────────────
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
    center=0, linewidths=0.5, square=True, cbar_kws={"shrink": 0.8}, ax=ax
)
ax.set_title("Correlation Heatmap — Numeric Features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Explicit track breakdown ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
counts = df["explicit"].value_counts()

axes[0].bar(counts.index.astype(str), counts.values,
            color=["#4C72B0", "#DD8452"], edgecolor="black")
axes[0].set_title("Explicit Track Count", fontweight="bold")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, str(v), ha="center", fontweight="bold")

axes[1].pie(counts.values, labels=counts.index.astype(str), autopct="%1.1f%%",
            colors=["#4C72B0", "#DD8452"], startangle=90)
axes[1].set_title("Proportion of Explicit Tracks", fontweight="bold")

plt.suptitle("Explicit Column Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()